In [1]:
# Use the following inside the jupyter terminal in sequence. This is for setting up the CPT VLLM model
"""
pip install uv
uv venv --python 3.12 --seed
source .venv/bin/activate
uv pip install vllm --torch-backend=auto
export TIKTOKEN_RS_CACHE_DIR="/lambda/nfs/NDS1"
vllm serve openai/gpt-oss-120b --gpu-memory-utilization 0.85

vllm serve openai/gpt-oss-120b --gpu-memory-utilization 0.9
vllm serve openai/gpt-oss-120b



vllm serve meta-llama/Llama-3.1-8B-Instruct
huggingface-cli login
"""

'\npip install uv\nuv venv --python 3.12 --seed\nsource .venv/bin/activate\nuv pip install vllm --torch-backend=auto\nexport TIKTOKEN_RS_CACHE_DIR="/lambda/nfs/NDS1"\nvllm serve openai/gpt-oss-120b --gpu-memory-utilization 0.85\n\nvllm serve openai/gpt-oss-120b --gpu-memory-utilization 0.9\nvllm serve openai/gpt-oss-120b\n\n\n\nvllm serve meta-llama/Llama-3.1-8B-Instruct\nhuggingface-cli login\n'

In [1]:
!pip install openai
!pip install distro
!pip install openpyxl
!pip install transformers
!pip install accelerate
!pip install faiss-cpu
!pip install tf-keras
!pip install anthropic
!pip install sentence_transformers
!pip install chardet

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import sys
sys.path.append("/home/ubuntu/.local/lib/python3.12/site-packages") 
sys.path.append("/usr/lib/python3/dist-packages")
sys.path.append('/lambda/nfs/NDS1/03_MedicalCoding/Endpoint_SourceCodes')
from openai import OpenAI
import os, chardet, time
import pandas as pd
from detection_fns_surgery_GPU import main_execute_single, main_execute_single_icd, main_execute_single_cpt, main_execute_single_hcpcs # GPU based
from DL_MODEL_ICD_CPT_combined import *  # Imports ICD and CPT related modules
# from detection_fns_surgery_CPU import main_execute_single, main_execute_single_icd, main_execute_single_cpt, main_execute_single_hcpcs # CPU based 
# from DL_MODEL_ICD_CPT_combined_CPU import * # Imports ICD and CPT related modules (CPU based)
import transformers, torchvision, accelerate, torch, requests, re, warnings, json, time, os, csv, openai
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
# from mybackend import *
from mybackend_DEV import *

/home/ubuntu/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/lambda/nfs/NDS1/03_MedicalCoding/Endpoint_SourceCodes/mybackend_DEV.py:611: SyntaxWarning: invalid escape sequence '\ '
  candidate_paring_kyphectomy_tethering =   """
/lambda/nfs/NDS1/03_MedicalCoding/Endpoint_SourceCodes/mybackend_DEV.py:643: SyntaxWarning: invalid escape sequence '\ '
  candidate_radicalresection =  """


mybackend_DEV loaded


In [3]:
import pandas as pd
pd.set_option('display.max_rows', None)      # show all rows
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.width', None)         # don't wrap columns
pd.set_option('display.max_colwidth', None)  # show full column contents

In [4]:
# Load DL Model   # Require transformers version 4.57.3
class ModelLoader:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(ModelLoader, cls).__new__(cls)
            cls._instance.load_models()
        return cls._instance
    def load_models(self):
            dl_object = DL_Model_ICD_CPT()
            self.model_ICD, self.LABELS_ICD,self.model_CPT, self.LABELS_CPT, self.tokenizer_common = dl_object.Create_DL_Model()
model_loader = ModelLoader()
print("DL Models loaded successfully")

# Load Finetuned Adapters and Databases
DF_All, DFA, DFP, ftmod, ftind = finetuned_adapter_loader_helper1()
DF_ALL_I, ftmod_i, ftind_i = finetuned_adapter_loader_helper3()
print("Finetuned loaded successfully")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /lambda/nfs/NDS1/03_MedicalCoding/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /lambda/nfs/NDS1/03_MedicalCoding/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CPT MODEL LOADED SUCCESSFULLY ON CPU


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /lambda/nfs/NDS1/03_MedicalCoding/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /lambda/nfs/NDS1/03_MedicalCoding/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ICD MODEL LOADED SUCCESSFULLY ON CPU
DL Models loaded successfully


/lambda/nfs/NDS1/03_MedicalCoding/Endpoint_SourceCodes/mybackend_DEV.py:611: SyntaxWarning: invalid escape sequence '\ '
  candidate_paring_kyphectomy_tethering =   """
/lambda/nfs/NDS1/03_MedicalCoding/Endpoint_SourceCodes/mybackend_DEV.py:643: SyntaxWarning: invalid escape sequence '\ '
  candidate_radicalresection =  """


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [6]:
############ Load the master tenet coding summary ############
mastersummary_path = "/lambda/nfs/NDS1/03_MedicalCoding/Demo_Tenet_charts/01_Master Tenet Coding Summary_03202025.xlsx"
mastersummarydf = pd.read_excel(mastersummary_path)

In [7]:
def process_chart_CPT_addon(chart_text,modelcode2,DFA):
    # print("Finding addon")
    final_addon_list = [] # We will store all the final list of addon codes here
    cpt_list = []; desc_list = []; candidates = []
    prefix = modelcode2[:3] # "113"
    subdf = DFA[DFA["CPT"].astype(str).str.startswith(prefix)] # DF
    cpt_list = subdf["CPT"].tolist(); 
    desc_list = subdf["Full Description"].tolist()
    if cpt_list:
        for tmp_cpt2, tmp_desc2 in zip(cpt_list,desc_list):
            joined_stat = tmp_cpt2 + ": " + tmp_desc2;  # 11300: Shaving asdfldj
            joined_stat =joined_stat.replace(";"," "); 
            candidates.append(joined_stat) # Example: ["11300:sadasd","14240:akjdhash"]
    if candidates: 
        candidates = "|".join(candidates)  # # Example: "11300:sadasd|14240:akjdhash"
        client = OpenAI(base_url="http://localhost:8000/v1",api_key="EMPTY")
        first_prompt = "You are an expert medical coder"
        second_prompt = """
        Your task is to choose the suitable CPT code(s) for the given text chart from among the given candidate.
        Rules:
        1. The candidate is arranged in the format <CPT>:<Full description>|<CPT>:<Full description>. Two candidates are separated by pipe (|), and there may be only one candidate also.
        2. See whether any of the candidate CPT can be billed by finding relevant context from the given text chart.
        3. These candidate are CPT addon codes and may be reported / repeated / billed n-times.
        4. Note that this addon code's primary code was already reported so your task is to simply check if it can be billed further or not.
        5. Do not give any other code beyond the given candidate CPT codes.
        6. Only give the CPT code strictly separated by comma (,).
        7. If you do not find any CPT code as necessary, just write "NOT_NEEDED" strictly.
        8. Finally, do not give me any explanation.
        The chart text is:
        """
        candidates = candidates.replace(",","").replace("\u202f", " ").replace("/", " ")
        prompt = second_prompt + "\n" + chart_text + "\n\n\n" + "The list of candidates are: " + candidates
        try:
            response = client.responses.create(model= "openai/gpt-oss-120b",
                                               instructions=first_prompt,
                                               input=prompt,reasoning={"effort": "medium"},temperature=0.0, max_output_tokens=20000)
            response = response.output_text
            if "NOT_NEEDED" in response:
                pass  # Just for reference and understanding
            else:
                llm_addons = response.split(",")
                final_addon_list = final_addon_list + llm_addons # Combine it together
        except:
            response =" "; print("Model did not generate")    
    else: 
        response = " "

    return final_addon_list

In [8]:
def process_chart_CPT_CT(third_prompt):
    print("Oh yes CT")
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

    first_prompt = "You are an expert medical coder"
    second_prompt = """
    Your task is to extract complete CT CPT procedure statements from the given CT report.
    Extract only the confirmed, billable CT procedures and list them as **bullet points**, with each bullet containing:
    **<Procedure Statement>: <CPT Code>** strictly put it in colon (:) spaced format. Strictly do not give any explanation.
        
    CT-SPECIFIC CODING RULES:
    1. **MODALITY MUST BE CT**: Only code procedures that are CT scans.
    2. **CONTRAST CODING CRITICAL**:
       - Code "without contrast" only if report explicitly states no contrast given
       - Code "with contrast" if IV/oral/rectal contrast documented
       - Code "without and with contrast" if both pre- and post-contrast images obtained
    3. **ANATOMICAL SPECIFICITY**:
       - Head/Neck/Brain: 70450-70498
       - Chest/Thorax: 71250-71275
       - Abdomen: 74150-74160
       - Pelvis: 72192-72194
       - Spine: 72125-72133
       - Extremities: 73200-73206, 73700-73706
       - Combined abdomen/pelvis: 74176-74178
    4. **CTA (CT ANGIOGRAPHY)**:
       - Use CTA codes only if angiography protocol mentioned: 70496, 71275, 74175, 75635
       - Requires arterial phase imaging documentation
    5. **POST-PROCESSING**:
       - +76376: 3D rendering (physician-performed)
       - +76377: 3D with interpretation
    6. **GUIDANCE**:
       - +77011-77013: CT guidance for procedures
    7. **CORONARY CTA**: 75571-75574 (cardiac CT with contrast)
    8. If the report mentions other modalities (MRI, US, X-ray) for comparison only, DO NOT code them.
    
    DO NOT INCLUDE:
    - Non-CT procedures (X-ray, MRI, Ultrasound, Nuclear Medicine)
    - Screening studies without diagnostic indication
    - Procedures not documented in report
    - Duplicate coding for same anatomical region
    
    FOR EACH PROCEDURE, CONFIRM:
    1. Is CT modality explicitly mentioned?
    2. Is contrast administration documented?
    3. Is anatomical region clearly specified?
    4. Is post-processing/3D mentioned?
    
    CT REPORT:
    """
    # sanitize text
    third_prompt = third_prompt.replace(",", "").replace("\u202f", " ").replace("/", " ")
    prompt = second_prompt + " " + third_prompt
    try:
        response = client.responses.create(
            model="openai/gpt-oss-120b",instructions=first_prompt,input=prompt,reasoning={"effort": "medium"},temperature=0.0,max_output_tokens=20000)
        response = response.output_text
        print("Stage-1: ", response)
             
        
    except Exception as e:
        print("Model did not generate", e)
        response = " "
    return response

In [9]:
# Final Code Selection and Validation
from sentence_transformers import SentenceTransformer, util
def choose_final(CPT_stats, Model_CPTs_raw, Model_CPTs_FT, DL_CPTs_raw, DL_CPTs_FT, DFP,chart_text,model_loader,ftmod):
    CPT_stats_split = CPT_stats.split("\n") # list format []
    Model_CPTs_raw_split = [item.strip() for item in Model_CPTs_raw.split(",")] # List 1
    Model_CPTs_FT_split = [item.strip() for item in Model_CPTs_FT.split(",")] # List 1
    DL_CPTs_raw_split = [item.strip() for item in DL_CPTs_raw.split(",")] # List 1
    DL_CPTs_FT_split = [item.strip() for item in DL_CPTs_FT.split(",")] # List 1

    DFP["CPT"] = DFP["CPT"].astype(str) # First convert to string properly
    
    final_statements=[]; final_model_CPT = []; score_list = []
    final_matching_not_matching = []; final_GOOD_BAD_SentTransformer = []; final_GOOD_BAD_LLM = [];

    ### First Finalize the perfect CPT codes
    for i in range(len(CPT_stats_split)):
        tmp_stat = CPT_stats_split[i]  # Statement to be tested
        tmp_Model_CPTs_raw_split = Model_CPTs_raw_split[i]
        tmp_Model_CPTs_FT_split = Model_CPTs_FT_split[i]
        tmp_DL_CPTs_raw_split = DL_CPTs_raw_split[i]
        tmp_DL_CPTs_FT_split = DL_CPTs_FT_split[i]

        candidates_lst = []
        df_match_tmp_Model_CPTs_raw_split = DFP[DFP["CPT"] == tmp_Model_CPTs_raw_split]
        if not df_match_tmp_Model_CPTs_raw_split.empty:
            for tmp_i in range(len(df_match_tmp_Model_CPTs_raw_split)):
                row_tmp_Model_CPTs_raw_split = df_match_tmp_Model_CPTs_raw_split.iloc[tmp_i]
                candidate_string1 = f"{row_tmp_Model_CPTs_raw_split['CPT']} : {row_tmp_Model_CPTs_raw_split['Full Description']}"
                candidates_lst.append(candidate_string1)    
            ## To choose the top as the candidate follow the following
            # row_tmp_Model_CPTs_raw_split = df_match_tmp_Model_CPTs_raw_split.iloc[0]
            # candidate_string1 = f"{row_tmp_Model_CPTs_raw_split['CPT']} : {row_tmp_Model_CPTs_raw_split['Full Description']}"
            # candidates_lst.append(candidate_string1)
            
       df_match_tmp_Model_CPTs_FT_split = DFP[DFP["CPT"] == tmp_Model_CPTs_FT_split]
        if not df_match_tmp_Model_CPTs_FT_split.empty:
            for tmp_i2 in range(len(df_match_tmp_Model_CPTs_FT_split)):
                row_tmp_Model_CPTs_FT_split = df_match_tmp_Model_CPTs_FT_split.iloc[tmp_i2]
                candidate_string2 = f"{row_tmp_Model_CPTs_FT_split['CPT']} : {row_tmp_Model_CPTs_FT_split['Full Description']}"
                candidates_lst.append(candidate_string2)
                
        # df_match_tmp_DL_CPTs_raw_split = DFP[DFP["CPT"] == tmp_DL_CPTs_raw_split]
        # if not df_match_tmp_DL_CPTs_raw_split.empty:
        #     row_tmp_DL_CPTs_raw_split = df_match_tmp_DL_CPTs_raw_split.iloc[0]
        #     candidate_string3 = f"{row_tmp_DL_CPTs_raw_split['CPT']} : {row_tmp_DL_CPTs_raw_split['Full Description']}"
        #     candidates_lst.append(candidate_string3)
        #     print(candidate_string3)    
        
       
        df_match_tmp_DL_CPTs_FT_split = DFP[DFP["CPT"] == tmp_DL_CPTs_FT_split]
        if not df_match_tmp_DL_CPTs_FT_split.empty:
            for tmp_i4 in range(len(df_match_tmp_DL_CPTs_FT_split)):
                row_tmp_DL_CPTs_FT_split = df_match_tmp_DL_CPTs_FT_split.iloc[tmp_i4]
                candidate_string4 = f"{row_tmp_DL_CPTs_FT_split['CPT']} : {row_tmp_DL_CPTs_FT_split['Full Description']}"
                candidates_lst.append(candidate_string4)
       
        candidates_lst = list(set(candidates_lst))
        candidates_lst_joined = "|".join(candidates_lst)

        client = OpenAI(base_url="http://localhost:8000/v1",api_key="EMPTY")
        first_prompt = "You are an expert medical coder"
        second_prompt = """
        Your task is to choose and give me the best suitable CPT code(s) for the given test procedure statement from among the given candidate(s).
        Rules:
        1. The candidate is arranged in the format <CPT>:<Full description>|<CPT>:<Full description>. Two candidates are separated by pipe (|), and there may be only one candidate also.
        2. Choose the best applicable CPT from among these candidates that can be used for billing perfectly.
        3. Do not give any other code beyond the given candidate CPT codes.
        4. Give only the CPT code <CPT>. Do not give me the description.
        4. Finally, do not give me any explanation.
        The test procedure statement is:
        """
        prompt = second_prompt + "\n" + tmp_stat + "\n\n\n" + "The list of candidates are: " + candidates_lst_joined
        try:
            response = client.responses.create(model= "openai/gpt-oss-120b",
                                               instructions=first_prompt,
                                               input=prompt,reasoning={"effort": "medium"},temperature=0.0, max_output_tokens=20000)
            response = response.output_text
            final_statements.append(tmp_stat)
            final_model_CPT.append(response)
        except: pass

    ## Second determine matching or notmatching, Good/Bad
    for i in range(len(CPT_stats_split)):
        tmp_stat = CPT_stats_split[i]  # Statement to be tested
        tmp_modelcodee = final_model_CPT[i] # Chosen Model CPT Code
        tmp_dlcodee = DL_CPTs_FT_split[i] # DL Code
        if tmp_modelcodee.strip() in tmp_dlcodee.strip(): final_matching_not_matching.append("MATCHED")
        else: final_matching_not_matching.append("NOT MATCHING")
        
        df_match_tmp_modelcodee = DFP[DFP["CPT"] == tmp_modelcodee]
        if not df_match_tmp_modelcodee.empty:
            row_tmp_modelcodee = df_match_tmp_modelcodee.iloc[0]
            fd_statement_modelcode = row_tmp_modelcodee['Full Description']
            ### Apply Sentence Transformer that does vector matching
            emb1 = ftmod.encode(tmp_stat, convert_to_tensor=True, normalize_embeddings=True)
            emb2 = ftmod.encode(fd_statement_modelcode, convert_to_tensor=True, normalize_embeddings=True)
            score = util.cos_sim(emb1, emb2).item(); score = round(score,2)
            score_list.append(str(score))
            if score >= 0.8: 
                final_GOOD_BAD_SentTransformer.append("GOOD")
            else: 
                final_GOOD_BAD_SentTransformer.append("BAD")
            
            ### Apply LLM Matching that does contextual matching
            result_code = tmp_modelcodee + ": " + fd_statement_modelcode
            
            client = OpenAI(base_url="http://localhost:8000/v1",api_key="EMPTY")
            first_prompt = "You are an expert medical coder"
            second_prompt = """
            Check whether the given CPT code is suitable for the given test procedure statement.
            Rules:
            1. If applicable strictly write as "GOOD SUITABLE", otherwise write "BAD NOT SUITABLE".
            2. Finally, do not give me any explanation. Do not give CPT code or the description.
            The test procedure statement is:
            """
            prompt = second_prompt + "\n" + tmp_stat + "\n\n\n" + "The CPT code is: " + result_code
            try:
                # response = client.responses.create(model= "openai/gpt-oss-120b",
                #                                    instructions=first_prompt,
                #                                    input=prompt,reasoning={"effort": "medium"},temperature=0.0, max_output_tokens=20000)
                # response = response.output_text
                response = "FUNCTIONALITY SUSPENDED CURRENTLY"
                if "GOOD" in response:
                    final_GOOD_BAD_LLM.append("GOOD")
                elif "BAD" in response:
                    final_GOOD_BAD_LLM.append("BAD")
                else:
                    final_GOOD_BAD_LLM.append("FUNCTIONALITY SUSPENDED CURRENTLY")
            except:
                final_GOOD_BAD_LLM.append("BAD")
        else:
            score_list.append("CODE NOT FOUND")
            final_GOOD_BAD_SentTransformer.append("CODE NOT FOUND")
            final_GOOD_BAD_LLM.append("CODE NOT FOUND")

    ###### Final operation before writing
    if final_statements:
        final_statements_str = "\n".join(final_statements)
        final_model_CPT_str = ",".join(final_model_CPT)
        score_list_str = ",".join(score_list)
        final_matching_not_matching_str  = ",".join(final_matching_not_matching)
        final_GOOD_BAD_SentTransformer_str =  ",".join(final_GOOD_BAD_SentTransformer)
        final_GOOD_BAD_LLM_str =  ",".join(final_GOOD_BAD_LLM)        
    else:
        final_statements_str = "";   final_model_CPT_str = ""; score_list_str =""  ;final_matching_not_matching_str = "";final_GOOD_BAD_SentTransformer_str = ""; final_GOOD_BAD_LLM_str = ""
    return final_statements_str, final_model_CPT_str, score_list_str, final_matching_not_matching_str, final_GOOD_BAD_SentTransformer_str, final_GOOD_BAD_LLM_str

def acquire_CPT_CT(chart_text,ftmod, ftind, DFP, DFA,model_loader):
    final_response_CPT = process_chart_CPT_CT(chart_text) # Let LLM generate statement and code first
    statements_list = []; # Generated by LLM
    statements_list2 = []; # Chosen by Finetuned LLM
    model_codes_list = [];  # Generated by LLM
    model_codes_list2 = []; # Chosen by Finetuned LLM
    DL_codes_list = []; # Applied LLM Generated statement to CPT DL model
    DL_codes_list2 = [] # Applied Finetune LLM's Generated statement to CPT DL model
    Addon_list = []
    for line in final_response_CPT.split("\n"):
        if ":" in line:
            parts = line.split(":", 1)
            stmt = parts[0].strip();  modelcode = parts[1].strip()
            stmt2,modelcode2 = finetuned_adapter_loader_helper2(stmt,ftmod, ftind, DFP) # Primary codes
            stmt = spinalvertebra_preprocessing(stmt) 
            # CPTcode=" "; CPTconfidence=" "; CPTstatement=" ";
            # CPTcode2=" "; CPTconfidence2=" "; CPTstatement2=" ";
            CPTcode, CPTconfidence, CPTstatement    = DL_testing_CPT(stmt, model_loader)
            CPTcode2, CPTconfidence2, CPTstatement2 = DL_testing_CPT(stmt2, model_loader)
            ### Get addon code
            addon = process_chart_CPT_addon(chart_text,modelcode2,DFA) # List format []
            statements_list.append(stmt);            statements_list2.append(stmt2)
            model_codes_list.append(modelcode);      model_codes_list2.append(modelcode2)
            DL_codes_list.append(CPTcode);           DL_codes_list2.append(CPTcode2)
            if addon:
                Addon_list = Addon_list + addon # Concat
    statements_str = "\n".join(statements_list);     statements_str2 = "\n".join(statements_list2)
    model_codes_str = ", ".join(model_codes_list);   model_codes_str2 = ", ".join(model_codes_list2)
    dl_codes_str = ", ".join(DL_codes_list);         dl_codes_str2 = ", ".join(DL_codes_list2)
    addon_codes_str = ", ".join(Addon_list)
    return final_response_CPT, statements_str, model_codes_str, model_codes_str2, dl_codes_str, dl_codes_str2, addon_codes_str

In [ ]:
#%%%%%%%%%%%
input_folder = r"/lambda/nfs/NDS1/03_MedicalCoding/Demo_Tenet_charts/All_Tenet_Charts/6 CT"
output_folder_CPT = r"/lambda/nfs/NDS1/03_MedicalCoding/Demo_Tenet_charts/DEV_output/CT_200_CPT.xlsx"
output_folder_ICD = r"/lambda/nfs/NDS1/03_MedicalCoding/Demo_Tenet_charts/DEV_output/CT_200_ICD.xlsx"

df_CPT = pd.DataFrame(columns=["Sr_no", "filename", "LLMresponse", 
                               "LLMstats", "LLM_CPT_raw", "LLM_CPT_FT", "DLcodesRaw", "DLcodesFT",
                               "LLM_stats_final", "LLM_CPT_final",
                               "MATCHING", "ST_SCORE", "GOOD_BAD_ST", "GOOD_BAD_LLM",
                               "ModelAddonCPTs", "Custcodes_CPT", "time_elapsed_sec"])
df_ICD = pd.DataFrame(columns=["Sr_no", "filename", "LLMresponse", "LLMstats", "LLMcodes", "DLcodes", "Custcodes_ICD", "time_elapsed_sec"])
files = os.listdir(input_folder)
# for i in range(len(files)):
for i in range(0,200):
    print(i)
    filename = files[i]
    if not filename.lower().endswith(".txt"):
        continue
    start_time = time.time()
    filename_without_txt = filename.replace(".txt", ""); print(filename_without_txt)
    cs_codes_cpt = acquire_CS_CPT(filename_without_txt,mastersummarydf) # Fetch Code Summary. # str format
    cs_codes_icd = acquire_CS_ICD(filename_without_txt,mastersummarydf) # Fetch Code Summary. # str format    
    chart_text = read_chart(input_folder, filename) # Load chart
    ####################### CPT Task #######################
    # 1: Acquire the statements and the primary codes
    response_CPT, CPT_stats, Model_CPTs_raw, Model_CPTs_FT, DL_CPTs_raw, DL_CPTs_FT, Model_addons = acquire_CPT_CT(chart_text,ftmod,ftind,DFP,DFA,model_loader)  # Acquire CPT codes # Strings
    final_statements_str, final_model_CPT_str, score_list_str, final_matching_not_matching_str, final_GOOD_BAD_SentTransformer_str, final_GOOD_BAD_LLM_str = choose_final(CPT_stats, Model_CPTs_raw, Model_CPTs_FT, DL_CPTs_raw, DL_CPTs_FT, DFP,chart_text,model_loader,ftmod) # Str
    # 2: Evaluate add-on codes 
    # final_codes = evaluate_additionalcodes(chart_text)
    final_codes = []
    final_codes_str = ""
    if final_codes: final_codes_str = ", ".join(final_codes)
    ####################### ICD Task #######################
    response_ICD = " "; ICD_stats = " "; Model_ICDs = " "; DL_ICDs = " ";
    # response_ICD, ICD_stats, Model_ICDs, DL_ICDs = acquire_ICD(chart_text,ftmod_i,ftind_i,DF_ALL_I,model_loader)  # Acquire ICD codes
    ########################################################
    cs_codes_str_cpt = cs_codes_cpt # Just for reference
    cs_codes_str_icd = cs_codes_icd # Just for reference
    time_elapsed = round(time.time() - start_time, 2)
    df_CPT.loc[len(df_CPT)] = [i, filename, response_CPT, 
                               CPT_stats, Model_CPTs_raw, Model_CPTs_FT, DL_CPTs_raw, DL_CPTs_FT, 
                               final_statements_str, final_model_CPT_str,
                               final_matching_not_matching_str, score_list_str, final_GOOD_BAD_SentTransformer_str, final_GOOD_BAD_LLM_str,
                               Model_addons , cs_codes_str_cpt, time_elapsed]
    df_ICD.loc[len(df_ICD)] = [i, filename, response_ICD, ICD_stats, Model_ICDs, DL_ICDs, cs_codes_str_icd, time_elapsed]
    df_CPT.to_excel(output_folder_CPT, index=False)
    df_ICD.to_excel(output_folder_ICD, index=False)
    # display(df_CPT)
    # print(df_CPT); # print(df_ICD); 
    print(f"[{i}] Processed: {filename}"); 
    # print("CPT_stats:\n",CPT_stats); print("Model_CPTs:",Model_CPTs); print("DL_CPTs:",DL_CPTs); 
    # print("Additional_CPTs:",final_codes_str); print("CS_Codes_CPT:",cs_codes_str_cpt)
    # print("ICD_stats:\n",ICD_stats); print("Model_ICDs:",Model_ICDs); print("DL_ICDs:",DL_ICDs); print("CS_Codes_ICD:",cs_codes_str_icd)
    print("\n...........................\n")
print("\n✔ All files processed and saved to:", output_folder_CPT)

0
100806934
Oh yes Neuro
Stage-1:  
Stage-2:  - Revision left carpal tunnel release: 64721
Stage-3:  - Revision left carpal tunnel release: 64721
[0] Processed: 100806934.txt

...........................

1
100813625
Oh yes Neuro
Stage-1:  * (No billable Neurology procedures identified in the provided documentation.)
Stage-2:  - No billable Neurology procedures identified in the provided documentation.: N/A
Stage-3:  - Left ulnar nerve decompression with anterior subcutaneous transposition at elbow: 64718  
- Left middle finger A1 pulley release (trigger finger): 26055  
- Left ring finger A1 pulley release (trigger finger): 26055  
- Left little finger A1 pulley release (trigger finger): 26055
[1] Processed: 100813625.txt

...........................

2
100824564
Oh yes Neuro
Stage-1:  
Stage-2:  -
Stage-3:  - Right open carpal tunnel release: 64721


In [ ]:
display(df_CPT)